# LC 739 — Daily Temperatures
**Difficulty:** Medium &nbsp;|&nbsp; **Category:** Stack
**Pattern:** Monotonic Decreasing Stack — Next Greater Element

<div style="border-left:4px solid purple; padding:10px 16px;
            background:#f5f0ff; margin-top:12px;">
<strong>Core Insight:</strong> Stack stores indices of
days still waiting for a warmer day. When a warmer
temperature arrives, pop each waiting index and
record the gap. Unresolved indices stay 0.
</div>

## Official Problem Statement

Given an array of integers `temperatures` represents
the daily temperatures, return an array `answer`
such that `answer[i]` is the number of days you
have to wait after the `i`th day to get a warmer
temperature. If there is no future day for which
this is possible, keep `answer[i] == 0` instead.

**Example 1:**
```
Input:  temperatures = [73,74,75,71,69,72,76,73]
Output: [1,1,4,2,1,1,0,0]
```
**Example 2:**
```
Input:  temperatures = [30,40,50,60]
Output: [1,1,1,0]
```
**Example 3:**
```
Input:  temperatures = [30,60,90]
Output: [1,1,0]
```

**Constraints:**
- `1 <= temperatures.length <= 10^5`
- `30 <= temperatures[i] <= 100`

## What This Is Actually Asking

For each day, look forward in time and find the
first day that is warmer than today.
Return how many days ahead that warmer day is.
If no warmer day ever comes, return 0 for that day.
The challenge: do it without a nested loop.

## Walk Through an Example by Hand

```
temps = [73, 74, 75, 71, 69, 72, 76, 73]
idx:      0   1   2   3   4   5   6   7

result = [0,  0,  0,  0,  0,  0,  0,  0]
stack  = []   (stores indices)

i=0 temp=73  stack empty -> push 0   stack=[0]
i=1 temp=74  74>temps[0]=73 -> pop 0  result[0]=1-0=1
             stack empty -> push 1   stack=[1]
i=2 temp=75  75>temps[1]=74 -> pop 1  result[1]=2-1=1
             stack empty -> push 2   stack=[2]
i=3 temp=71  71<temps[2]=75 -> push  stack=[2,3]
i=4 temp=69  69<temps[3]=71 -> push  stack=[2,3,4]
i=5 temp=72  72>temps[4]=69 -> pop 4  result[4]=5-4=1
             72>temps[3]=71 -> pop 3  result[3]=5-3=2
             72<temps[2]=75 -> push  stack=[2,5]
i=6 temp=76  76>temps[5]=72 -> pop 5  result[5]=6-5=1
             76>temps[2]=75 -> pop 2  result[2]=6-2=4
             stack empty -> push 6   stack=[6]
i=7 temp=73  73<temps[6]=76 -> push  stack=[6,7]

End: stack [6,7] stay 0 (no warmer day)
Answer: [1, 1, 4, 2, 1, 1, 0, 0]
```

## The Picture

```
temps = [73, 74, 75, 71, 69, 72, 76, 73]

Think of the stack as a "waiting room".
Days sit in the waiting room until a warmer
day arrives to resolve them.

Day 0 (73) -> waiting room: [0]
Day 1 (74) -> 74 > 73: resolve day 0  gap=1
           -> waiting room: [1]
Day 2 (75) -> 75 > 74: resolve day 1  gap=1
           -> waiting room: [2]
Day 3 (71) -> 71 < 75: join wait   [2,3]
Day 4 (69) -> 69 < 71: join wait   [2,3,4]
Day 5 (72) -> 72 > 69: resolve 4   gap=1
           -> 72 > 71: resolve 3   gap=2
           -> 72 < 75: join wait   [2,5]
Day 6 (76) -> 76 > 72: resolve 5   gap=1
           -> 76 > 75: resolve 2   gap=4  <-
           -> waiting room: [6]
Day 7 (73) -> 73 < 76: join wait   [6,7]
End -> 6,7 never resolved -> stay 0

Stack is always DECREASING in temperature
(monotonic). Each element enters/exits ONCE -> O(n).
```

## When To Use This Pattern

- When you see **"next greater element to the right"**,
  think **monotonic decreasing stack of indices**
- When each element needs to find something to its
  right, think **stack stores unresolved elements;
  new elements resolve them**
- When all inner loops together run at most n times,
  think **amortized O(n) — each index pushed/popped
  at most once**
- When elements left in the stack at the end get 0,
  think **initialise result array to all zeros**

## The Approach

Initialise the result array to all zeros and an
empty stack.
Walk through each temperature: while the stack is
not empty and the current temperature is greater
than the temperature at the index on top of the
stack, pop that index and set its result to the
difference between the current index and the popped
index.
Push the current index onto the stack and continue.
Anything left in the stack at the end already has
result 0.

In [1]:
from typing import List  # type hints for the solution

In [2]:
def test_harness(func):
    tests = [
        # (temperatures, expected)
        ([73,74,75,71,69,72,76,73], [1,1,4,2,1,1,0,0]),
        ([30,40,50,60],             [1,1,1,0]),
        ([30,60,90],                [1,1,0]),
        ([90,60,30],                [0,0,0]),  # descending
        ([30],                      [0]),       # single day
        ([70,70,70],                [0,0,0]),  # all equal
        ([55,38,80,15,22,90,59,76], [2,1,3,1,1,0,1,0]),
    ]

    passed = 0
    for i, (temps, expected) in enumerate(tests):
        result = func(temps[:])
        ok = result == expected
        status = "PASSED" if ok else "FAILED"
        if ok:
            passed += 1
        print(
            f"Test {i+1}: {status} | "
            f"temps={temps} | "
            f"expected={expected} | got={result}"
        )

    print(f"\n{passed}/{len(tests)} tests passed")

In [14]:
def dailyTemperatures(temperatures: List[int]) -> List[int]:
    """
    Return days to wait for a warmer temperature each day.

    Init result to zeros. Walk with a stack of unresolved
    indices. For each temp: while stack top's temp <
    current, pop and set result[popped] = i - popped.
    Push current index. Remaining stack indices stay 0.

    Time:  O(n) — each index pushed and popped once
    Space: O(n) — stack holds at most n indices
    """
    # Intially isntantiate a zero List for every day
    #intially instantiate an empty stack to house indexes of a montonic decreasing  indexes of tempratures
    #for a temprature bigger than stack top temprature , evict value and populate result with current index - popped index
    # print(dailyTemperatures([90,60,30]))     # [0,0,0]  it is a decreasing montonic stack
    
    res = [0] * len(temperatures)
    stack = [] # monotonic decreasing stack
    for i, temp in enumerate(temperatures):
        while stack and temperatures[stack[-1]] < temp:
            idx = stack.pop()
            res[idx] = i - idx
        stack.append(i)
    return res
"""
[1, 1, 4, 2, 1, 1, 0, 0]
[1, 1, 1, 0]
[0, 0, 0]
[0]
Test 1: PASSED | temps=[73, 74, 75, 71, 69, 72, 76, 73] | expected=[1, 1, 4, 2, 1, 1, 0, 0] | got=[1, 1, 4, 2, 1, 1, 0, 0]
Test 2: PASSED | temps=[30, 40, 50, 60] | expected=[1, 1, 1, 0] | got=[1, 1, 1, 0]
Test 3: PASSED | temps=[30, 60, 90] | expected=[1, 1, 0] | got=[1, 1, 0]
Test 4: PASSED | temps=[90, 60, 30] | expected=[0, 0, 0] | got=[0, 0, 0]
Test 5: PASSED | temps=[30] | expected=[0] | got=[0]
Test 6: PASSED | temps=[70, 70, 70] | expected=[0, 0, 0] | got=[0, 0, 0]
Test 7: PASSED | temps=[55, 38, 80, 15, 22, 90, 59, 76] | expected=[2, 1, 3, 1, 1, 0, 1, 0] | got=[2, 1, 3, 1, 1, 0, 1, 0]

7/7 tests passed
"""


# Quick debug — run this cell while building
print(dailyTemperatures([73,74,75,71,69,72,76,73]))
# [1,1,4,2,1,1,0,0]
print(dailyTemperatures([30,40,50,60]))  # [1,1,1,0]
print(dailyTemperatures([90,60,30]))     # [0,0,0]
print(dailyTemperatures([30]))           # [0]
test_harness(dailyTemperatures)

[1, 1, 4, 2, 1, 1, 0, 0]
[1, 1, 1, 0]
[0, 0, 0]
[0]
Test 1: PASSED | temps=[73, 74, 75, 71, 69, 72, 76, 73] | expected=[1, 1, 4, 2, 1, 1, 0, 0] | got=[1, 1, 4, 2, 1, 1, 0, 0]
Test 2: PASSED | temps=[30, 40, 50, 60] | expected=[1, 1, 1, 0] | got=[1, 1, 1, 0]
Test 3: PASSED | temps=[30, 60, 90] | expected=[1, 1, 0] | got=[1, 1, 0]
Test 4: PASSED | temps=[90, 60, 30] | expected=[0, 0, 0] | got=[0, 0, 0]
Test 5: PASSED | temps=[30] | expected=[0] | got=[0]
Test 6: PASSED | temps=[70, 70, 70] | expected=[0, 0, 0] | got=[0, 0, 0]
Test 7: PASSED | temps=[55, 38, 80, 15, 22, 90, 59, 76] | expected=[2, 1, 3, 1, 1, 0, 1, 0] | got=[2, 1, 3, 1, 1, 0, 1, 0]

7/7 tests passed


In [ ]:
def dailyTemperatures(temperatures: List[int]) -> List[int]:
    """
    Return days to wait for a warmer temperature each day.

    Init result to zeros. Walk with a stack of unresolved
    indices. For each temp: while stack top's temp <
    current, pop and set result[popped] = i - popped.
    Push current index. Remaining stack indices stay 0.

    Time:  O(n) — each index pushed and popped once
    Space: O(n) — stack holds at most n indices
    """
    pass


# Quick debug — run this cell while building
print(dailyTemperatures([73,74,75,71,69,72,76,73]))
# [1,1,4,2,1,1,0,0]
print(dailyTemperatures([30,40,50,60]))  # [1,1,1,0]
print(dailyTemperatures([90,60,30]))     # [0,0,0]
print(dailyTemperatures([30]))           # [0]

In [ ]:
# Uncomment and run when solution is ready
# test_harness(dailyTemperatures)

## Complexity

| Approach | Time | Space |
|---|---|---|
| Brute force — nested scan | O(n²) | O(1) |
| Monotonic stack | O(n) | O(n) |

The stack is O(n) because each index enters and
exits at most once — the while loop's total work
across all iterations is bounded by n.

## Real World Connection

At Citi, the capacity planning system needs to know
for each server: how many days until CPU utilisation
first exceeds today's reading — the first sign of
sustained growth that may require provisioning.
This is the Daily Temperatures problem mapped to
telemetry: temperatures → CPU readings, warmer →
higher utilisation, gap → days until first breach.
The monotonic stack scans the 90-day rolling window
across 6,000 servers in O(n) instead of the O(n²)
nested scan that would miss the Prophet forecasting
pipeline's latency budget.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra